<a href="https://colab.research.google.com/github/tatahonon/cvNotebooks/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass, field
from functools import lru_cache
from typing import Dict, Iterable, List, Optional, Sequence, Tuple


Allocation = Tuple[int, ...]
Action = Optional[int]


@dataclass(frozen=True)
class State:
    p1: Allocation
    p2: Allocation
    b1: int
    b2: int
    turn: int

    @property
    def n(self) -> int:
        return len(self.p1)


def legal_actions(state: State) -> Tuple[Action, ...]:
    if state.b1 == 0 and state.b2 == 0:
        return tuple()
    if state.turn == 1:
        return tuple(range(state.n)) if state.b1 > 0 else (None,)
    return tuple(range(state.n)) if state.b2 > 0 else (None,)


def apply_action(state: State, action: Action) -> State:
    if action is None:
        return State(state.p1, state.p2, state.b1, state.b2, 3 - state.turn)

    if state.turn == 1:
        p1 = list(state.p1)
        p1[action] += 1
        return State(tuple(p1), state.p2, state.b1 - 1, state.b2, 2)

    p2 = list(state.p2)
    p2[action] += 1
    return State(state.p1, tuple(p2), state.b1, state.b2 - 1, 1)


def terminal_margin(state: State, values: Sequence[int]) -> int:
    margin = 0
    for a, b, v in zip(state.p1, state.p2, values):
        if a > b:
            margin += v
        elif a < b:
            margin -= v
    return margin


def normalize_margin(margin: int, values: Sequence[int]) -> float:
    return margin / sum(values)


def exact_value_small(values: Tuple[int, ...], b1: int, b2: int) -> Tuple[int, int]:
    @lru_cache(None)
    def value(p1: Allocation, p2: Allocation, rb1: int, rb2: int, turn: int) -> Tuple[int, Action]:
        state = State(p1, p2, rb1, rb2, turn)
        if rb1 == 0 and rb2 == 0:
            return terminal_margin(state, values), None
        actions = legal_actions(state)
        if actions == (None,):
            nxt = apply_action(state, None)
            result, _ = value(nxt.p1, nxt.p2, nxt.b1, nxt.b2, nxt.turn)
            return result, None
        if turn == 1:
            best = (-10**9, None)
            for action in actions:
                nxt = apply_action(state, action)
                result, _ = value(nxt.p1, nxt.p2, nxt.b1, nxt.b2, nxt.turn)
                if result > best[0]:
                    best = (result, action)
            return best
        best = (10**9, None)
        for action in actions:
            nxt = apply_action(state, action)
            result, _ = value(nxt.p1, nxt.p2, nxt.b1, nxt.b2, nxt.turn)
            if result < best[0]:
                best = (result, action)
        return best

    result, action = value((0,) * len(values), (0,) * len(values), b1, b2, 1)
    assert action is not None
    return action, result


@dataclass
class Node:
    state: State
    parent: Optional["Node"] = None
    action_from_parent: Action = None
    visits: int = 0
    value_sum: float = 0.0
    children: Dict[Action, "Node"] = field(default_factory=dict)

    def untried_actions(self) -> List[Action]:
        return [a for a in legal_actions(self.state) if a not in self.children]


def uct_select_child(node: Node, exploration: float) -> Node:
    assert node.visits > 0
    parent_log = math.log(node.visits)
    best_score = -float("inf")
    best_children: List[Node] = []
    for child in node.children.values():
        if child.visits == 0:
            score = float("inf")
        else:
            avg = child.value_sum / child.visits
            if node.state.turn == 2:
                avg = -avg
            score = avg + exploration * math.sqrt(parent_log / child.visits)
        if score > best_score:
            best_score = score
            best_children = [child]
        elif score == best_score:
            best_children.append(child)
    return random.choice(best_children)


def rollout(state: State, values: Sequence[int]) -> float:
    while state.b1 or state.b2:
        actions = legal_actions(state)
        action = random.choice(actions)
        state = apply_action(state, action)
    return normalize_margin(terminal_margin(state, values), values)


def mcts_action(
    root_state: State,
    values: Sequence[int],
    iterations: int = 2000,
    exploration: float = 1.41,
) -> Action:
    actions = legal_actions(root_state)
    if len(actions) == 1:
        return actions[0]

    root = Node(root_state)
    for _ in range(iterations):
        node = root
        path = [node]

        while node.untried_actions() == [] and legal_actions(node.state):
            node = uct_select_child(node, exploration)
            path.append(node)

        untried = node.untried_actions()
        if untried:
            action = random.choice(untried)
            child = Node(apply_action(node.state, action), parent=node, action_from_parent=action)
            node.children[action] = child
            node = child
            path.append(node)

        reward = rollout(node.state, values)
        for visited in path:
            visited.visits += 1
            visited.value_sum += reward

    def child_mean(child: Node) -> float:
        return child.value_sum / child.visits if child.visits else 0.0

    if root_state.turn == 1:
        return max(root.children.values(), key=child_mean).action_from_parent
    return min(root.children.values(), key=child_mean).action_from_parent


def greedy_p2_action(state: State, values: Sequence[int]) -> Action:
    actions = legal_actions(state)
    if actions == (None,):
        return None
    tied_or_losing = [a for a in actions if state.p2[a] <= state.p1[a]]
    candidates = tied_or_losing or list(actions)
    return max(candidates, key=lambda a: (values[a], -a))


def play_game(
    values: Sequence[int],
    b1: int,
    b2: int,
    p2_agent: str,
    iterations: int,
) -> State:
    state = State((0,) * len(values), (0,) * len(values), b1, b2, 1)
    while state.b1 or state.b2:
        actions = legal_actions(state)
        if actions == (None,):
            state = apply_action(state, None)
            continue
        if state.turn == 1:
            action = mcts_action(state, values, iterations=iterations)
        elif p2_agent == "greedy":
            action = greedy_p2_action(state, values)
        elif p2_agent == "mcts":
            action = mcts_action(state, values, iterations=iterations)
        else:
            raise ValueError(f"Unknown Player 2 agent: {p2_agent}")
        state = apply_action(state, action)
    return state


def summarize_games(
    values: Sequence[int],
    b1: int,
    b2: int,
    p2_agent: str,
    games: int = 100,
    iterations: int = 2000,
    seed: int = 20260414,
) -> None:
    random.seed(seed)
    margins: List[int] = []
    terminal_boards: Dict[Tuple[Allocation, Allocation], int] = {}
    p1_alloc_totals = [0] * len(values)
    p2_alloc_totals = [0] * len(values)

    for _ in range(games):
        terminal = play_game(values, b1, b2, p2_agent, iterations)
        margin = terminal_margin(terminal, values)
        margins.append(margin)
        terminal_boards[(terminal.p1, terminal.p2)] = terminal_boards.get((terminal.p1, terminal.p2), 0) + 1
        for i, x in enumerate(terminal.p1):
            p1_alloc_totals[i] += x
        for i, x in enumerate(terminal.p2):
            p2_alloc_totals[i] += x

    wins = sum(1 for m in margins if m > 0)
    ties = sum(1 for m in margins if m == 0)
    losses = games - wins - ties
    avg_margin = sum(margins) / games
    most_common = sorted(terminal_boards.items(), key=lambda item: (-item[1], item[0]))[:8]
    avg_p1 = [x / games for x in p1_alloc_totals]
    avg_p2 = [x / games for x in p2_alloc_totals]

    print(f"Scenario: P1 MCTS vs P2 {p2_agent}, B1={b1}, B2={b2}, games={games}, iterations={iterations}, seed={seed}")
    print(f"W/T/L for Player 1: {wins}/{ties}/{losses}")
    print(f"Average margin for Player 1: {avg_margin:.2f}")
    print(f"Margin counts: {dict(sorted({m: margins.count(m) for m in set(margins)}.items()))}")
    print(f"Average P1 allocation: {avg_p1}")
    print(f"Average P2 allocation: {avg_p2}")
    print("Most common terminal boards:")
    for (p1, p2), count in most_common:
        print(f"  {count:3d}x  P1={p1}, P2={p2}, margin={terminal_margin(State(p1, p2, 0, 0, 1), values)}")
    print()


def main() -> None:
    small_action, small_value = exact_value_small((3, 2), 2, 2)
    print("Exact small game:")
    print(f"  optimal first action: state {small_action + 1}")
    print(f"  guaranteed margin: {small_value}")
    print()

    values = (10, 7, 5, 3)
    summarize_games(values, 12, 10, "greedy")
    summarize_games(values, 12, 10, "mcts")
    summarize_games(values, 10, 10, "mcts")


if __name__ == "__main__":
    main()


Exact small game:
  optimal first action: state 1
  guaranteed margin: 0

Scenario: P1 MCTS vs P2 greedy, B1=12, B2=10, games=100, iterations=2000, seed=20260414
W/T/L for Player 1: 100/0/0
Average margin for Player 1: 17.00
Margin counts: {17: 100}
Average P1 allocation: [11.0, 1.0, 0.0, 0.0]
Average P2 allocation: [10.0, 0.0, 0.0, 0.0]
Most common terminal boards:
  100x  P1=(11, 1, 0, 0), P2=(10, 0, 0, 0), margin=17

Scenario: P1 MCTS vs P2 mcts, B1=12, B2=10, games=100, iterations=2000, seed=20260414
W/T/L for Player 1: 100/0/0
Average margin for Player 1: 16.80
Margin counts: {11: 3, 15: 1, 17: 96}
Average P1 allocation: [8.21, 3.65, 0.11, 0.03]
Average P2 allocation: [7.18, 2.75, 0.07, 0.0]
Most common terminal boards:
   36x  P1=(9, 3, 0, 0), P2=(8, 2, 0, 0), margin=17
   29x  P1=(8, 4, 0, 0), P2=(7, 3, 0, 0), margin=17
   19x  P1=(7, 5, 0, 0), P2=(6, 4, 0, 0), margin=17
   11x  P1=(10, 2, 0, 0), P2=(9, 1, 0, 0), margin=17
    2x  P1=(4, 4, 3, 1), P2=(2, 6, 2, 0), margin=11
    